# Chargement des données

In [ ]:
! bash init.sh

In [ ]:
import duckdb
%load_ext sql

In [ ]:
%sql duckdb:///istex-search-metadata-small.db

## Echantillonnage

### Hosts

In [ ]:
%%sql
SELECT
  *
FROM
  hosts USING SAMPLE 1000 (reservoir);

## Records

In [ ]:
%%sql
SELECT
  *
FROM
  records USING SAMPLE 1000 (reservoir);

## Metadata

In [ ]:
%%sql
SELECT
  *
FROM
  metadata USING SAMPLE 1000 (reservoir);

# Analyse

In [ ]:
%%sql
SELECT
  *
FROM
  hosts USING SAMPLE 1000 (reservoir);

In [ ]:
%%sql
SELECT
  COUNT(*) AS "Documents créés par la chaine d'ingestion Istex"
FROM
  (
    SELECT
      "path"
    FROM
      metadata
    WHERE
      original = FALSE
    UNION ALL
    SELECT
      "path"
    FROM
      fulltext
    WHERE
      original = FALSE
  );

In [ ]:
%%sql

WITH cte_extension_fulltext AS (
    SELECT 'fulltext' AS "type", LOWER(string_split(path, '.')[-1]) AS extension FROM fulltext
),
cte_extension_metadata AS (
    SELECT 'metadata' AS "type", LOWER(string_split(path, '.')[-1]) AS extension FROM metadata
),
cte_extension AS (
    SELECT * FROM cte_extension_fulltext
    UNION ALL
    SELECT * FROM cte_extension_metadata
)
    
SELECT extension, type, COUNT(*) AS "Nombre de fichiers" FROM cte_extension
GROUP BY extension, type;

# Analyse statistique à faire sur les fichiers OCR produits par Istex

* OCR / par décénnies
* OCR par disciplines
* OCR par langue

# Données à récupérer dans l'API Istex

arkIstex
genre -> du document
language -> du document
publicationDate
accessCondition.value (on pourra inférer le accessCondition.contentType au besoin)
corpusName
metadata ->  ark, path, mime, orignal
fulltext -> ark, path, mime, orignal

## Analyse fichier PDF

10 % du fichier PDF arrodi à l'entier supérieur. Selection aléatoire pour éviter de tomber sur des pages d'annexes

### Au niveau de chaque page (ou d'un échantillon de page)

* Liste des images et page sur laquelle a été trouvée l'image + taille de l'image + taille de la page
* Présence de texte transparent -> compter le nombre de caractères (pour faire des ratios) ? 
* Présence de texte pas transparent -> compter le nombre de caractères
* Présence de texte très fragmenté (Tj) 
* Taille de police unique


In [ ]:
%config SqlMagic.displaylimit = 100

In [ ]:
%%sql

SELECT arkIstex  FROM fulltext WHERE PATH like '%.ocr';